# Tugas 1: Information Retrieval - Text Preprocessing Pipeline

**Mata Kuliah:** Information Retrieval  
**Topik:** Text Preprocessing (Loading, Tokenization, Normalization, Stopword Removal, Stemming, Lemmatization)  
**Dataset:** YouTube Comments Dataset (`YoutubeCommentsDataSet.csv`)  
**Custom Datasets:** `normalization_dict.csv` & `stopwords.txt`  
**Output:** Explanatory Tabular Report (`preprocessing_results_table.txt`)

---

### 📌 Preprocessing Stages Overview
1. **Load Data Text:** Membaca subset sample dataset teks dari file CSV.
2. **Normalization:** Case folding, pembersihan URL/emoticon/tanda baca, dan standarisasi singkatan/slang/kontraksi menggunakan kamus normalisasi (`normalization_dict.csv`).
3. **Tokenization:** Memecah string kalimat menjadi token-token kata.
4. **Stop Word Removal:** Menghapus kata-kata umum/noise yang tidak memiliki bobot informasi penting menggunakan NLTK Stopwords dan custom stopwords dataset (`stopwords.txt`).
5. **Stemming:** Memotong imbuhan kata ke bentuk dasar menggunakan algoritma *Porter Stemmer* & *Snowball Stemmer*.
6. **Lemmatization:** Mengembalikan kata ke bentuk lema kamus yang tepat berdasarkan konteks *Part-of-Speech* (POS) menggunakan *WordNet Lemmatizer*.
7. **Explanatory Table Export:** Menyusun dan mengekspor tabel perbandingan setiap kolom proses ke dalam file `.txt` lengkap dengan penjelasannya.

## ⚙️ Step 0: Environment Setup & Library Installation
Install and import all required packages, and download NLTK corpora.

In [ ]:
# Install dependencies (for Google Colab or local environment)
!pip install -q nltk pandas tabulate matplotlib seaborn

In [ ]:
import os
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

import nltk
# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk import pos_tag

print("✅ Setup Completed! NLTK and all libraries are ready.")

## 📂 Step 1: Load Data & Custom Datasets (Sample Only)

Kita memuat sample beberapa baris teks dari:
1. `dataset/YoutubeCommentsDataSet.csv` (Dataset komentar YouTube)
2. `dataset/normalization_dict.csv` (Dataset kamus normalisasi slang/kontraksi)
3. `dataset/stopwords.txt` (Dataset stopword kustom)

In [ ]:
# File paths (Supports Google Colab and Local Folder)
DATASET_PATH = 'dataset/YoutubeCommentsDataSet.csv' if os.path.exists('dataset/YoutubeCommentsDataSet.csv') else 'YoutubeCommentsDataSet.csv'
NORM_DICT_PATH = 'dataset/normalization_dict.csv' if os.path.exists('dataset/normalization_dict.csv') else 'normalization_dict.csv'
STOPWORDS_PATH = 'dataset/stopwords.txt' if os.path.exists('dataset/stopwords.txt') else 'stopwords.txt'

# 1. Load Main Text Dataset (Sample 5 records)
SAMPLE_LIMIT = 5
if os.path.exists(DATASET_PATH):
    df_full = pd.read_csv(DATASET_PATH)
    df = df_full.dropna(subset=['Comment']).head(SAMPLE_LIMIT).copy()
    print(f"✅ Dataset loaded! Taking sample of {len(df)} records.")
else:
    print("⚠️ Dataset file not found in current path. Using built-in sample comments for demonstration.")
    sample_data = {
        'Comment': [
            "lets not forget that apple pay in 2014 required a brand new iphone in order to use it a significant portion of apples user base wasnt able to use it",
            "here in nz 50 of retailers don’t even have contactless credit card machines like paywave which support apple pay they don’t like the high fees",
            "i will forever acknowledge this channel with the help of your lessons and ideas explanations now its quite helpful while youll just sit at your comfort",
            "whenever i go to a place that doesn’t take apple pay doesn’t happen too often it’s such a drag between contactless covid habits and my getting the apple card",
            "apple pay is so convenient secure and easy to use i used it while at the korean and japanese airports no need for physical credit cards"
        ],
        'Sentiment': ['neutral', 'negative', 'positive', 'negative', 'positive']
    }
    df = pd.DataFrame(sample_data)

df[['Comment']].head()

In [ ]:
# 2. Load Custom Normalization Dataset
normalization_dict = {}
if os.path.exists(NORM_DICT_PATH):
    norm_df = pd.read_csv(NORM_DICT_PATH)
    normalization_dict = dict(zip(norm_df['slang'].astype(str).str.lower(), norm_df['formal'].astype(str).str.lower()))
    print(f"✅ Loaded {len(normalization_dict)} normalization pairs from '{NORM_DICT_PATH}'")
else:
    normalization_dict = {
        'u': 'you', 'ur': 'your', 'r': 'are', 'dont': 'do not', 'doesnt': 'does not',
        'didnt': 'did not', 'cant': 'cannot', 'wont': 'will not', 'wanna': 'want to',
        'gonna': 'going to', 'tbh': 'to be honest', 'idk': 'i do not know', 'pls': 'please',
        'thx': 'thanks', 'rn': 'right now', 'psu': 'power supply unit', 'lmg': 'linus media group'
    }

# 3. Load Custom Stopwords Dataset
custom_stopwords = set()
if os.path.exists(STOPWORDS_PATH):
    with open(STOPWORDS_PATH, 'r', encoding='utf-8') as f:
        custom_stopwords = set(line.strip().lower() for line in f if line.strip())
    print(f"✅ Loaded {len(custom_stopwords)} custom stopwords from '{STOPWORDS_PATH}'")
else:
    custom_stopwords = {'video', 'channel', 'subscribe', 'youtube', 'like', 'guys', 'comment'}

# Combine with NLTK standard English stopwords
all_stopwords = set(stopwords.words('english')).union(custom_stopwords)
print(f"✅ Total active stopwords: {len(all_stopwords)}")

## 🧹 Step 2: Normalization (Text Cleaning & Slang Replacement)

**Penjelasan:**
- Mengubah huruf menjadi *lowercase* (*case folding*).
- Menghapus URL, tag HTML, emoji, dan karakter non-ASCII.
- Mengganti singkatan, slang internet, dan kontraksi (misal: `dont` -> `do not`, `ltt` -> `linus tech tips`, `psu` -> `power supply unit`).
- Menghapus tanda baca (*punctuation*) dan merapikan spasi berlebih.

In [ ]:
def normalize_text(text, norm_dict=normalization_dict, remove_numbers=False):
    if not isinstance(text, str):
        text = str(text)

    # Lowercase
    text = text.lower()
    # Remove URLs & HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    # Clean curly quotes & emojis
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = text.encode('ascii', 'ignore').decode('utf-8')
    
    # Slang & Contraction Replacement
    words = text.split()
    normalized_words = [norm_dict.get(w, w) for w in words]
    text = " ".join(normalized_words)
    
    # Remove punctuation
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    
    if remove_numbers:
        text = re.sub(r'\d+', ' ', text)
        
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 🔠 Step 3: Tokenization

**Penjelasan:**
Memecah teks string hasil normalisasi menjadi daftar unit-unit kata (*tokens*) menggunakan NLTK `word_tokenize`.

In [ ]:
def tokenize_text(text):
    try:
        return word_tokenize(text)
    except Exception:
        return text.split()

## 🚫 Step 4: Stop Word Removal

**Penjelasan:**
Menghapus kata-kata fungsional (seperti *the, is, at, which, on, video, channel, subscribe*) yang sering muncul tetapi tidak membawa informasi spesifik pembeda dalam Information Retrieval.

In [ ]:
def remove_stopwords(tokens, stop_words_set=all_stopwords):
    return [token for token in tokens if token not in stop_words_set and len(token) > 1]

## 🌲 Step 5: Stemming

**Penjelasan:**
Memotong imbuhan kata (*affixes*) dengan aturan morfologis heuristik *Porter Stemmer* untuk mendapatkan akar kata dasar.

In [ ]:
porter = PorterStemmer()

def stem_tokens(tokens):
    return [porter.stem(w) for w in tokens]

## 📚 Step 6: Lemmatization (POS-Aware)

**Penjelasan:**
Mengembalikan kata ke lema kamus (*canonical base form*) dengan mempertimbangkan *Part-of-Speech* (kelas kata seperti Kata Kerja, Kata Benda, Kata Sifat) menggunakan NLTK `WordNetLemmatizer`.

In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_tokens(tokens):
    if not tokens:
        return []
    try:
        tagged_tokens = pos_tag(tokens)
        return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged_tokens]
    except Exception:
        return [lemmatizer.lemmatize(word) for word in tokens]

## 📊 Step 7: Generate Preprocessing Table with Column Explanations & Output to TXT

Menjalankan seluruh tahapan pada sampel teks dan menyusun tabel terformat dengan penjelasan pada setiap kolomnya ke file `preprocessing_results_table.txt`.

In [ ]:
# Process each sample
table_rows = []
for idx, row in df.iterrows():
    orig_text = str(row['Comment'])
    norm_text = normalize_text(orig_text)
    tokens = tokenize_text(norm_text)
    stops_removed = remove_stopwords(tokens)
    stemmed = stem_tokens(stops_removed)
    lemmatized = lemmatize_tokens(stops_removed)
    
    table_rows.append([
        f"Sample {idx + 1}",
        orig_text,
        norm_text,
        ", ".join(tokens),
        ", ".join(stops_removed),
        ", ".join(stemmed),
        ", ".join(lemmatized)
    ])

table_headers = [
    "Sample",
    "Original Text\n(Raw Input)",
    "1. Normalization\n(Lowercased, Slang Replaced,\nPunctuation/URLs Removed)",
    "2. Tokenization\n(Split into Individual\nWord Tokens)",
    "3. Stopword Removal\n(Filtered Noise &\nCommon Non-Informative Words)",
    "4. Stemming\n(Porter Stemmer Root\nAffix Chopping)",
    "5. Lemmatization\n(WordNet POS-Aware\nDictionary Base Form)"
]

# Format Table using ASCII Grid
formatted_table = tabulate(table_rows, headers=table_headers, tablefmt="grid", maxcolwidths=[10, 25, 25, 25, 25, 25, 25])

# Build Full Explanatory TXT Report
txt_report = f"""========================================================================================================================
                      INFORMATION RETRIEVAL (TUGAS 1) - TEXT PREPROCESSING PIPELINE REPORT
========================================================================================================================
Dataset Source : YoutubeCommentsDataSet.csv (Sample subset: {len(df)} comments)
Dictionaries   : normalization_dict.csv (Slang/Contractions), stopwords.txt (Custom + NLTK Stopwords)
========================================================================================================================

COLUMN DEFINITIONS & PREPROCESSING EXPLANATIONS:
------------------------------------------------------------------------------------------------------------------------
1. [Original Text]
   - Raw, unprocessed comment text directly loaded from the dataset.

2. [Normalization]
   - Transformasi teks ke bentuk standar:
     * Case Folding: Mengubah semua karakter menjadi huruf kecil (lowercase).
     * Noise Removal: Menghapus URL, tag HTML, emoji, dan karakter non-ASCII.
     * Slang & Contraction Replacement: Mengganti kata tidak baku / singkatan menggunakan kamus normalisasi
       (contoh: 'u' -> 'you', 'dont' -> 'do not', 'psu' -> 'power supply unit', 'lmg' -> 'linus media group').
     * Punctuation Removal: Menghapus simbol dan tanda baca yang tidak esensial.

3. [Tokenization]
   - Pemecahan string kalimat yang telah dinormalisasi menjadi unit-unit token kata individual menggunakan NLTK word_tokenize.

4. [Stopword Removal]
   - Pemfilteran kata-kata umum (seperti 'the', 'is', 'in', 'at', 'which') serta noise spesifik YouTube/media sosial
     ('video', 'channel', 'subscribe', 'like') yang tidak memiliki bobot informasi pembeda dalam Information Retrieval.

5. [Stemming]
   - Pemotongan afiks/imbuhan kata secara heuristik menggunakan algoritma Porter Stemmer untuk menghasilkan akar kata
     (contoh: 'retailers' -> 'retail', 'required' -> 'requir').

6. [Lemmatization]
   - Pengembalian kata ke bentuk lema kamus yang tepat secara morfologis dengan mempertimbangkan kelas kata (Part-of-Speech / POS Tagging)
     menggunakan WordNet Lemmatizer (contoh: 'studies' -> 'study', 'better' -> 'good/well', 'went' -> 'go').
========================================================================================================================

TABULAR RESULTS FOR PROCESSED SAMPLES:
------------------------------------------------------------------------------------------------------------------------
{formatted_table}

========================================================================================================================
"""

# Save Report to TXT file
output_txt_filename = 'preprocessing_results_table.txt'
with open(output_txt_filename, 'w', encoding='utf-8') as f:
    f.write(txt_report)

print(formatted_table)
print(f"\n✅ Report table saved successfully to '{output_txt_filename}'")

## 🔍 Step 8: Stemming vs. Lemmatization Matrix Comparison

In [ ]:
test_words = ["running", "studies", "better", "went", "universities", "bought", "contactless", "retailers", "remorse"]
comparison_table = []

for word in test_words:
    p_stem = porter.stem(word)
    pos = get_wordnet_pos(pos_tag([word])[0][1])
    pos_lemma = lemmatizer.lemmatize(word, pos)
    comparison_table.append([word, p_stem, pos_lemma, pos])

print(tabulate(comparison_table, headers=["Original Word", "Porter Stemmer", "POS Lemmatizer", "POS Tag"], tablefmt="grid"))